In [1]:
import torch
import ray
import pandas as pd
from client import FLClient
from models import GCN, GAT
from server import Server
from utils import load_config
from dataprocessingset import load_processed_data, load_processed_data_with_hop, load_facebook_data
from feature_propagation import load_with_feature_prop

In [2]:
def initialize_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")

def load_data(num_clients, beta, data_type='normal', hop=None):
    if data_type == 'facebook':
        return load_facebook_data(num_clients=num_clients, beta=beta)
    elif data_type == 'hop':
        return load_processed_data_with_hop(num_clients=num_clients, beta=beta, hop=hop)
    elif data_type == 'feature_prop':
        return load_with_feature_prop(num_clients=num_clients, beta=beta, hop=hop)
    return load_processed_data(num_clients=num_clients, beta=beta)

def init_model(model_type, num_features, num_classes):
    if model_type == "GCN":
        return GCN(num_features, 16, num_classes)
    elif model_type == "GAT":
        return GAT(num_features, 16, num_classes)
    raise ValueError("Unsupported model type")

In [3]:
def run_training_rounds(server, num_rounds):
    """
    result is made up of loss and accuracy tuples, one for each client so result alone is list of 10 items
    train_results is a list of 10 lists, each containing the loss and accuracy for each client
    """
    train_results = []
    for i in range(num_rounds):
        result = server.train_clients(i)
        print(f"Round {i+1} is complete")
        train_results.append(result)
    return train_results, server.evaluate_clients()


def evaluate_models(server, dataset, test_data):
    criterion = torch.nn.CrossEntropyLoss()
    test_results = server.test_global_model(dataset)
    client_test_results = [client.test.remote(test) for client, test in zip(server.clients, test_data)]
    client_results = ray.get(client_test_results)
    print(f"The final global test results: {test_results}")
    return test_results, sum(client_results) / len(client_results)



In [4]:

def run_experiments(num_clients, beta, cfg, model_type="GAT", data_type = 'normal'):
    DEVICE = initialize_device()
  
    if data_type == 'feature_prop':
        data, cora_dataset, clients_data, test_data = load_data(num_clients, beta, data_type, cfg['hop'])
    else:
        data, cora_dataset, clients_data = load_data(num_clients, beta, data_type)
    test_data = clients_data if 'test_data' not in locals() else test_data
    
    data = data.to(DEVICE)
    print(cora_dataset, len(clients_data))

    model = init_model(model_type, cora_dataset.num_features, cora_dataset.num_classes)
    clients = [FLClient.remote(data, cora_dataset, i, cfg, model_type) for i, data in enumerate(clients_data)]
    server = Server(clients, model)

    return server, cora_dataset, clients_data, test_data




In [5]:
# Importing necessary libraries and modules
import torch
import ray
import pandas as pd
from client import FLClient
from models import GCN, GAT
from server import Server
from utils import load_config
from dataprocessingset import load_processed_data, load_processed_data_with_hop, load_facebook_data
from feature_propagation import load_with_feature_prop

# Initialize the device
def initialize_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load data based on the specified type
def load_data(num_clients, beta, data_type='normal', hop=None):
    if data_type == 'facebook':
        return load_facebook_data(num_clients=num_clients, beta=beta)
    elif data_type == 'hop':
        return load_processed_data_with_hop(num_clients=num_clients, beta=beta, hop=hop)
    elif data_type == 'feature_prop':
        return load_with_feature_prop(num_clients=num_clients, beta=beta, hop=hop)
    return load_processed_data(num_clients=num_clients, beta=beta)

# Initialize the model based on the type
def init_model(model_type, num_features, num_classes):
    if model_type == "GCN":
        return GCN(num_features, 16, num_classes)
    elif model_type == "GAT":
        return GAT(num_features, 16, num_classes)
    raise ValueError("Unsupported model type")

# Running training rounds
def run_training_rounds(server, num_rounds):
    train_results = []
    for i in range(num_rounds):
        result = server.train_clients(i)
        print(f"Round {i+1} is complete")
        train_results.append(result)
    return train_results, server.evaluate_clients()

# Evaluating the models
def evaluate_models(server, dataset, test_data):
    criterion = torch.nn.CrossEntropyLoss()
    test_results = server.test_global_model(dataset)
    client_test_results = [client.test.remote(test) for client, test in zip(server.clients, test_data)]
    client_results = ray.get(client_test_results)
    print(f"The final global test results: {test_results}")
    return test_results, sum(client_results) / len(client_results)

# Running the experiments
def run_experiments(num_clients, beta, cfg, model_type="GAT", data_type='normal'):
    DEVICE = initialize_device()

    if data_type == 'feature_prop':
        data, cora_dataset, clients_data, test_data = load_data(num_clients, beta, data_type, cfg['hop'])
    else:
        data, cora_dataset, clients_data = load_data(num_clients, beta, data_type)
        test_data = clients_data if 'test_data' not in locals() else test_data
    
    data = data.to(DEVICE)
    print(cora_dataset, len(clients_data))

    model = init_model(model_type, cora_dataset.num_features, cora_dataset.num_classes)
    clients = [FLClient.remote(data, cora_dataset, i, cfg, model_type) for i, data in enumerate(clients_data)]
    server = Server(clients, model)

    return server, cora_dataset, clients_data, test_data

# Ensure the modules are in the path
import sys
import os
sys.path.append(os.getcwd())

# Run the main script
ray.init()
cfg = load_config("conf/base.yaml")
server, cora_dataset, clients_data, test_data = run_experiments(10, 100, cfg, 'GAT', 'normal')
train_results, eval_results = run_training_rounds(server, cfg['num_rounds'])
# final_results = evaluate_models(server, cora_dataset, test_data)
ray.shutdown()


2024-08-07 20:27:49,955	INFO worker.py:1770 -- Started a local Ray instance.


Cora() 10


(FLClient pid=2794) 2024-08-07 20:28:04,084 - INFO - Epoch   0| Train Loss: 2.007| Train Accuracy: 0.211
(FLClient pid=2794) 2024-08-07 20:28:04,210 - INFO - Epoch   2| Validation Loss: 1.919, Validation Accuracy: 0.194


Round 1 is complete
Round 2 is complete
Round 3 is complete
Round 4 is complete
Round 5 is complete
Round 6 is complete
Round 7 is complete
Round 8 is complete
Round 9 is complete
Round 10 is complete
Round 11 is complete
Round 12 is complete
Round 13 is complete
Round 14 is complete
Round 15 is complete


(FLClient pid=2793) 2024-08-07 20:28:09,194 - INFO - Epoch   0| Train Loss: 0.598| Train Accuracy: 1.000 [repeated 450x across cluster] (Ray deduplicates logs by default. Set RAY_DEDUP_LOGS=0 to disable log deduplication, or see https://docs.ray.io/en/master/ray-observability/user-guides/configure-logging.html#log-deduplication for more options.)
(FLClient pid=2793) 2024-08-07 20:28:09,298 - INFO - Epoch   2| Validation Loss: 1.418, Validation Accuracy: 0.519 [repeated 150x across cluster]


Round 16 is complete
Round 17 is complete
Round 18 is complete
Round 19 is complete
Round 20 is complete
Round 21 is complete
Round 22 is complete
Round 23 is complete
Round 24 is complete
Round 25 is complete
Round 26 is complete
Round 27 is complete
Round 28 is complete
Round 29 is complete
Round 30 is complete
Round 31 is complete
Round 32 is complete
Round 33 is complete
Round 34 is complete


(FLClient pid=2828) 2024-08-07 20:28:13,358 - INFO - Epoch   2| Train Loss: 0.180| Train Accuracy: 1.000 [repeated 569x across cluster]
(FLClient pid=2828) 2024-08-07 20:28:13,364 - INFO - Epoch   2| Validation Loss: 1.391, Validation Accuracy: 0.574 [repeated 189x across cluster]


In [ ]:
ray.shutdown()